# 04 CPSC Plan-Level Analysis

Purpose: use the optional CPSC files for plan/contract-level opportunity analysis. This is deeper than MA SCP because it includes contract and plan identifiers.

Implementation decision: read the large CPSC enrollment CSVs in chunks. That avoids loading multi-gigabyte expanded CSV content into memory.

In [ ]:
from __future__ import annotations

import json
import re
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import PROCESSED_DIR, TABLE_DIR, FIGURE_DIR, CPSC_DIR, PMPM_PROXY_REVENUE

for path in [PROCESSED_DIR, TABLE_DIR, FIGURE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")

## Chunked CPSC Aggregation

Why: CPSC enrollment files are large. The notebook aggregates each monthly file to contract/plan/state/county level and writes compact outputs.

In [ ]:
def month_from_name(path: Path) -> str:
    match = re.search(r"(20\d{2}-\d{2})", path.name)
    if not match:
        raise ValueError(path.name)
    return match.group(1)


def find_csv(zf: zipfile.ZipFile, contains: str) -> str:
    matches = [n for n in zf.namelist() if contains.lower() in n.lower() and n.lower().endswith(".csv")]
    if not matches:
        raise ValueError(f"No CSV containing {contains}")
    return matches[0]


monthly_rows = []
contract_rows = []
zip_paths = sorted(CPSC_DIR.glob("*.zip"))

for zip_path in zip_paths:
    month = month_from_name(zip_path)
    with zipfile.ZipFile(zip_path) as zf:
        contract_csv = find_csv(zf, "Contract_Info")
        contract = pd.read_csv(zf.open(contract_csv), dtype=str, keep_default_na=False, encoding="cp1252")
        contract["report_month_label"] = month
        contract_rows.append(contract)

        enrollment_csv = find_csv(zf, "Enrollment_Info")
        chunks = pd.read_csv(
            zf.open(enrollment_csv),
            dtype=str,
            keep_default_na=False,
            encoding="cp1252",
            chunksize=250_000,
        )
        for chunk in chunks:
            chunk["Enrollment"] = pd.to_numeric(chunk["Enrollment"].str.replace(",", "", regex=False), errors="coerce")
            chunk["report_month_label"] = month
            grouped = (
                chunk.groupby(["report_month_label", "Contract Number", "Plan ID", "State", "County", "FIPS State County Code"], as_index=False)
                .agg(observed_enrollment=("Enrollment", "sum"), source_rows=("Enrollment", "size"))
            )
            monthly_rows.append(grouped)

cpsc_enrollment = pd.concat(monthly_rows, ignore_index=True)
cpsc_enrollment = (
    cpsc_enrollment.groupby(["report_month_label", "Contract Number", "Plan ID", "State", "County", "FIPS State County Code"], as_index=False)
    .agg(observed_enrollment=("observed_enrollment", "sum"), source_rows=("source_rows", "sum"))
)
cpsc_contract = pd.concat(contract_rows, ignore_index=True)

cpsc_enrollment.to_parquet(PROCESSED_DIR / "cpsc_contract_plan_county_monthly.parquet", index=False)
cpsc_contract.to_parquet(PROCESSED_DIR / "cpsc_contract_info_monthly.parquet", index=False)
print(f"CPSC enrollment rows after aggregation: {len(cpsc_enrollment):,}")
print(f"CPSC contract rows: {len(cpsc_contract):,}")

## Contract/Plan Growth Summary

Why: this identifies high-growth plans and contracts for RCM sales targeting.

In [ ]:
cpsc_enrollment["report_month"] = pd.PeriodIndex(cpsc_enrollment["report_month_label"], freq="M").to_timestamp()
plan_monthly = (
    cpsc_enrollment.groupby(["report_month", "report_month_label", "Contract Number", "Plan ID"], as_index=False)
    .agg(observed_enrollment=("observed_enrollment", "sum"), counties=("FIPS State County Code", "nunique"))
    .sort_values(["Contract Number", "Plan ID", "report_month"])
)

plan_growth = (
    plan_monthly.groupby(["Contract Number", "Plan ID"], as_index=False)
    .agg(
        first_month=("report_month_label", "first"),
        last_month=("report_month_label", "last"),
        months_present=("report_month_label", "nunique"),
        first_enrollment=("observed_enrollment", "first"),
        last_enrollment=("observed_enrollment", "last"),
        counties=("counties", "max"),
    )
)
plan_growth["absolute_growth"] = plan_growth["last_enrollment"] - plan_growth["first_enrollment"]
plan_growth["growth_pct"] = np.where(plan_growth["first_enrollment"] > 0, plan_growth["absolute_growth"] / plan_growth["first_enrollment"], np.nan)

latest_contract = cpsc_contract.sort_values("report_month_label").groupby(["Contract ID", "Plan ID"], as_index=False).tail(1)
plan_growth = plan_growth.merge(
    latest_contract[["Contract ID", "Plan ID", "Organization Name", "Plan Name", "Plan Type", "Parent Organization"]],
    left_on=["Contract Number", "Plan ID"],
    right_on=["Contract ID", "Plan ID"],
    how="left",
).sort_values(["absolute_growth", "last_enrollment"], ascending=False)

plan_monthly.to_parquet(PROCESSED_DIR / "cpsc_plan_monthly.parquet", index=False)
plan_growth.to_csv(TABLE_DIR / "cpsc_plan_growth.csv", index=False)
display(plan_growth.head(25))

## CPSC Visual Results

Elements: bars show absolute growth; color shows plan type where available. These are optional deeper sales leads beyond the national forecast.

In [ ]:
top_plans = plan_growth.head(20).copy()
top_plans["plan_label"] = top_plans["Contract Number"] + "-" + top_plans["Plan ID"]
fig = px.bar(top_plans, x="plan_label", y="absolute_growth", color="Plan Type", hover_name="Plan Name", title="Top CPSC Plans by Absolute Enrollment Growth")
fig.show()

plan_growth_scatter = plan_growth.replace([np.inf, -np.inf], np.nan).dropna(subset=["growth_pct"]).head(500).copy()
plan_growth_scatter["abs_growth_for_marker"] = plan_growth_scatter["absolute_growth"].abs().clip(lower=1)
fig = px.scatter(
    plan_growth_scatter,
    x="last_enrollment",
    y="growth_pct",
    size="abs_growth_for_marker",
    hover_name="Plan Name",
    color="Plan Type",
    title="CPSC Plan Scale vs Growth",
)
fig.show()